# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL: [https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Metadata is accessible as an object
md = dataset.metadata

print("---- Dataset Overview ----")
print(f"Name: {md.name}")
print(f"Description: {md.description}")
print(f"Identifier: {md.identifier}")
print(f"CiteAs: {md.citeAs}")
print(f"Date Published: {md.datePublished}")
print(f"Version: {md.version}")
print(f"License: {md.license}")
print("Keywords:")
pprint.pprint(md.keywords)

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the dataset's Croissant schema to enumerate available record sets and their fields by `@id`. All references use the `@id` property to ensure consistency.

In [ ]:
# mlcroissant exposes record sets via .record_sets property
record_sets = dataset.record_sets

print("---- Record Sets in Dataset ----")
for rs in record_sets:
    print(f"Record Set Name: {rs.name}")
    print(f"Record Set @id: {rs['@id']}")
    print("Fields:")
    for field in rs.fields:
        print(f"  Field Name: {field.name}")
        print(f"  Field @id: {field['@id']}")
        print(f"  Data Type: {getattr(field, 'dataType', 'N/A')}")
    print("-------------------------")
# Optionally display sample records from first record set
if record_sets:
    sample_rs_id = record_sets[0]['@id']
    print(f"Sample records from Record Set @id={sample_rs_id}:")
    for i, record in enumerate(dataset.records(record_set=sample_rs_id)):
        print(record)
        if i > 2:
            break

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview above.

In [ ]:
# Extract all data from all record sets
dfs = {}
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Loading data from record set IDs:")
for rs_id in record_set_ids:
    print(f"- {rs_id}")

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dfs[rs_id] = pd.DataFrame(records)

# Example: Display first record set's columns and head
if record_set_ids:
    main_rs_id = record_set_ids[0]
    print(f"DataFrame columns for Record Set @id={main_rs_id}:")
    print(dfs[main_rs_id].columns.tolist())
    display(dfs[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records by criteria, normalizing numeric fields, and grouping data by attributes. Reference fields/columns only via their exact `@id` where applicable.

*Example: Filter age > 50, normalize, group by sex.*

In [ ]:
# Choose main record set and relevant fields by `@id` (replace with actual IDs, shown below as examples)
# You can inspect the DataFrame columns to find the actual field IDs
main_rs_id = record_set_ids[0]
df = dfs[main_rs_id]

# Let's print columns for reference
print("Available columns in main DataFrame:")
for col in df.columns:
    print(col)

# For demonstration, let's assume:
# - age field: '@id': 'cr:age' (replace with real @id from overview)
# - sex field: '@id': 'cr:sex' (replace with real @id)

# If the dataset uses different IDs, substitute accordingly:
numeric_field_id = None
group_field_id = None
for col in df.columns:
    lower = col.lower()
    if ('age' in lower and numeric_field_id is None):
        numeric_field_id = col
    if ('sex' in lower or 'gender' in lower) and group_field_id is None:
        group_field_id = col

if numeric_field_id is None:
    numeric_field_id = df.columns[0] # fallback
if group_field_id is None:
    group_field_id = df.columns[1] # fallback

print(f"Numeric field selected (@id): {numeric_field_id}")
print(f"Group field selected (@id): {group_field_id}")

# Filtering, normalizing, grouping
threshold = 50
if numeric_field_id in df.columns:
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    display(filtered_df.head())

    # Normalize numeric field
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by categorical field
    if group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (mean of {numeric_field_id}):")
        display(grouped_df)

## 5. Visualization

Visualize distributions and relationships. Example: show histogram of age and bar plot of group counts.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for numeric field
if numeric_field_id in df.columns and pd.api.types.is_numeric_dtype(df[numeric_field_id]):
    plt.figure(figsize=(8,5))
    sns.histplot(df[numeric_field_id], bins=15, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

# Bar plot for group field
if group_field_id in df.columns:
    plt.figure(figsize=(7,5))
    sns.countplot(y=df[group_field_id])
    plt.title(f"Counts of {group_field_id}")
    plt.xlabel("Count")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion

We loaded and explored the FAIR^2 dataset using `mlcroissant`, reviewed metadata, identified available record sets and fields by `@id`, loaded records, and performed basic EDA. This approach enables reproducible, standardized analysis of clinical and molecular characteristics in cancer survivor datasets. Key numeric and categorical fields can be referenced directly by their `@id`s. For further analysis, refine filters and visualizations based on the clinical hypothesis and available field @ids.